# Anomaly Detection Notebook: Chapter 1
## Date Started:   8 August 2022
## Latest Update: 13 June 2024

In [ ]:
! pip list | grep python

## 0. Define Metadata / Main Variables

In [ ]:
# Define the name of the target class column here instead of manually typing it out everywhere
target_class_name = 6

# Fill in the names of what you want to call the 0 and 1 class
labels = ['inliers', 'outliers']

# Add directory as a string
rawdataDirectory = "../01-Data/Raw/"
dataDirectory = "../01-Data/"
dataFile = "thyroid.mat"

# Any exported artifacts will have this date
export_date = '202406'

### Key Variables
#### thyroidX - Individual test results
#### thyroidY - Final diagnostic result - Presence or absence of cancer 

## 1. Download data to directory - Done

## 2. Import relevant packages

In [ ]:
import sklearn 
import scipy
from scipy import io
import numpy as np

import matplotlib 
%matplotlib inline

import pandas as pd
import dask

from scipy.io.arff import loadarff
import scipy.io as sio

from collections import Counter
from sklearn.preprocessing import MinMaxScaler

from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

from sklearn.metrics import accuracy_score, classification_report,confusion_matrix 
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_score


## 3. Load Dataset

In [ ]:
dataFile = rawdataDirectory + dataFile
print (dataFile)

In [ ]:
# Load to dictionary
thyroidD = scipy.io.loadmat(dataFile)

## 4. Extract X and Y and Concatenate

### 4.1 Explore Dataset as Dictionary

In [ ]:
# No of key-value pairs
len(thyroidD)

In [ ]:
# "Iterating over a dictionary produces its' keys"
# The numeric data are stored in the values "X" and "y"
# Note: The value for X is a list of lists for diagnostic information
#       The value for y is a list of lists (length 1) giving the final diagnosis
[i for i in thyroidD]

In [ ]:
# Loop through both keys and values, by using the items() function:
for i, j in thyroidD.items():
    print(i, j)
# thyroidD

In [ ]:
thyroidHdr = thyroidD["__header__"]
thyroidVrsn = thyroidD["__version__"]
thyroidGlobals = thyroidD["__globals__"]

print("thyroidHdr = ", thyroidHdr,"\n",
      "thyroidVrsn = ", thyroidVrsn,"\n",
      "thyroidGlobals = ", thyroidGlobals,"\n")

In [ ]:
items = list(thyroidD.items())
for key, value in items:
    print(f"Key: {key}, Value: {value}")

### 4.2 Extract X and y

In [ ]:
#thyroid_X = pd.DataFrame(thyroidD["X"])
#print(type(thyroid_X))
#thyroid_X

In [ ]:
# Store the features and target objects in their own variables (numpy arrays)
#         for easy retreival
thyroid_X=thyroidD['X']
thyroid_y=thyroidD['y']

thyroid_X.shape, thyroid_y.shape

### 4.3 Concatenate the numpy arrays and Inspect

In [ ]:
# Convert numpy arrays to a dataframe
#   with the target values in the final column
thyroidDF=pd.DataFrame((np.concatenate((thyroid_X, thyroid_y), axis=1)))

pd.DataFrame.info(thyroidDF)

#### Inspect rows

In [ ]:
# Random sample of 5 records
thyroidDF.sample(5)

Lets check the head & tail to make sure there is nothing going on at the last row or the header

In [ ]:
thyroidDF.head(3)

In [ ]:
thyroidDF.tail(3)

#### No trouble with loading the data - both head and tail are clean

### Summary Stats

In [ ]:
thyroidDF.describe()
# Looks like all the numbers are between 0 and 1

### Unique Value Checking

In [ ]:
# Original code from recommended solution. Failed previously 
#  due to there being a repeated column name of "0"
#  SO use np concatenate rather than pd to avoid col name duplication
for col in thyroidDF.columns:
    print(col, len(thyroidDF[col].unique()))

#### All of the columns have a relatively small number of duplicate values. Except the last one. As the final diagnosis this should always be 0 or 1 

#### Identifying Bad Columns

In [ ]:
def find_bad_columns_function(dataframe):
    '''
    Args: dataframe for which there maybe columns of concern that need to be fixed or deleted
    
    Logic: Find the columns that have 
    Null values
    blanks in the strings
    quasi constant/constant values defined by less than 1% variance
    
    Returns: 4 lists containing those features that have nulls, blanks, constant values throughout for numerical and categorical
    
    '''
    
    ###### Finding Null Values
    null_col_list = dataframe.columns[dataframe.isna().any()].tolist()
    
    print('Identified {} features with atleast one null'.format(
        len(null_col_list)))

    ###### Finding Blank Spaces in the object column
    # Non-obvious nulls such as blanks: The line items where there are spaces 
    blank_space_col_list = []
    object_columns = dataframe.select_dtypes(include=['object']).columns

    for col in object_columns:
        if sum(dataframe[col]==' '):
            blank_space_col_list.append(col)

    print('Identified {} features with at least one blank space'.format(
        len(blank_space_col_list)))
    
    ####### Finding Quasi Constant/Constant Value in numerical columns
    # Lets remove the variables that have more than 99% of their values as the same 
    # ie their standard deviation is less than 1 %
    
    numeric_df = dataframe._get_numeric_data()
    constant_numeric_col_list = [col for col in numeric_df.columns if numeric_df[col].std()<0.01]

    print('Identified {} numeric features that have quasi-constant values'.format(
        len(constant_numeric_col_list)))
    
    # We use a separate logic for the non-numerical variables because if you have closely varying float values
    # then the below code snippet wont pick it up
    
    ###### Finding Quasi Constant/Constant non_numeric value
    constant_non_numeric_col_list = []
    
    # Find the columns that are not in numeric_df
    non_numeric_col_set = set(dataframe.columns) - set(numeric_df.columns)   

    for col in non_numeric_col_set:
        categorical_mode_value = (dataframe[col].mode().values)[0]
        fractional_presence = sum(dataframe[col]==categorical_mode_value)/len(dataframe) 
    
        if fractional_presence > 0.99:
            constant_non_numeric_col_list.append(col)
            
    print('Identified {} non-numeric features that have quasi-constant values'.format(
        len(constant_non_numeric_col_list)))
    
    return null_col_list, blank_space_col_list, constant_numeric_col_list, constant_non_numeric_col_list

In [ ]:
# use the above custom function to figure out the if there are any columns we need to be concerned about
null_col_list, blank_space_col_list, constant_numeric_col_list, \
constant_non_numeric_col_list = find_bad_columns_function(thyroidDF)

### 4.4 Data Cleansing
#### No bad columns
#### No null values
#### No categorical values to encode (one-hot for stat analysis)

### 4.5 Export DataFrame as csv

In [ ]:
dataDirectory = "../01-Data/Processed/"
csvFile = "thyroid.csv"
csvdataFile = dataDirectory + csvFile
thyroidDF.to_csv(csvdataFile, index = False) # Does not export the index column